In [ ]:
import os
import logging
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

# 커스텀 모듈 임포트 (기존 코드 유지)
from lora_scratch import freeze_model, ExtendedModel

# 로깅 설정
logging.basicConfig(level=logging.INFO)

# 환경 변수 설정
os.environ['HF_HOME'] = '/data1/aman/programs/'
os.environ["TRANSFORMERS_CACHE"] = '/data1/aman/programs/'
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ['TORCH_USE_CUDA_DSA'] = "1"

# 1. 데이터셋 로드 및 전처리
logging.info("데이터셋을 로드하고 분할합니다...")
dataset = pd.read_csv('data/train_llama_formatted.csv')
hf_dataset = Dataset.from_pandas(dataset)

# 컬럼 이름을 'text'로 변경 (불필요한 map 연산 제거)
hf_dataset = hf_dataset.rename_column("data", "text")

# 학습(Train)과 평가(Eval) 데이터셋으로 분리 (예: 90% 학습, 10% 평가)
split_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# 2. 모델 및 토크나이저 로드
logging.info("모델과 토크나이저를 준비합니다...")
model_id = "microsoft/phi-1_5"
phi_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, cache_dir='/data1/aman/programs/')
phi_tokenizer.pad_token = phi_tokenizer.eos_token
phi_tokenizer.padding_side = "right"

phi_base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # 메모리 절약을 위해 bfloat16으로 로드
    trust_remote_code=True,
    cache_dir='/data1/aman/programs/'
)

# 토크나이저와 모델의 vocab 크기 동기화
if phi_base_model.config.vocab_size != len(phi_tokenizer):
    phi_tokenizer.add_tokens([phi_tokenizer.unk_token] * (phi_base_model.config.vocab_size - len(phi_tokenizer)))
    phi_base_model.resize_token_embeddings(len(phi_tokenizer))

# 3. 커스텀 LoRA 적용
logging.info("커스텀 LoRA 모듈을 적용합니다...")
phi_lora_model = ExtendedModel(phi_base_model)
freeze_model(phi_lora_model)

# 4. 학습 설정 (TrainingArguments)
output_dir = "/data1/aman/programs/output_model" # 체크포인트가 섞이지 않도록 하위 폴더 지정 권장
num_train_epochs = 10

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True, # 메모리 절약을 위해 활성화 (누락된 부분 추가)
    optim="paged_adamw_32bit",
    save_steps=200,              # 0 대신 적절한 스텝마다 저장하도록 수정 (선택 사항)
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=True,                   # Ampere(RTX 3000번대) 이상 GPU인 경우 True 권장 (구형이면 fp16=True)
    fp16=False,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    group_by_length=True,
    evaluation_strategy="epoch", # 에폭마다 검증 수행
    save_strategy="epoch",       # 에폭마다 모델 저장
)

# 5. SFTTrainer 초기화
trainer = SFTTrainer(
    model=phi_lora_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,   # 평가 데이터셋 추가
    dataset_text_field="text",
    max_seq_length=512,          # OOM(메모리 초과) 방지를 위해 최대 길이 제한 지정
    tokenizer=phi_tokenizer,
    args=training_arguments,
    packing=False,
)

# 6. 학습 시작 및 최종 평가
logging.info("학습을 시작합니다...")
trainer.train()

logging.info("최종 모델 평가를 진행합니다...")
eval_metrics = trainer.evaluate()
logging.info(f"최종 평가 결과: {eval_metrics}")

# (선택) 최종 모델 저장
trainer.save_model(os.path.join(output_dir, "final_lora_model"))
logging.info("학습이 성공적으로 완료되었습니다!")